# Домашняя работа 6.

## Предисловие.
- Применение модели.
- Анализ доступных данных.
- Определение целевой переменной.
- Определение метрик качества модели.

Мы знаем, что начальными этапами построения ML-модели являются определение способа приенение модели, анализ доступных данных и определение целевой переменной. Однако в этой домашке мы не задаёмся этими вопросами, так как нам сразу подсказали и применение модели для максимизации прибыли, и целевую переменную, и даже данные предоставили, и указали цену ошибок разного рода: за ошибку первого рода мы просто не получим никакой прибыли, но и потерь никаких не понесём; за ошибку второго рода мы получаем штраф, равный 0.75 * desired_credit_limit. За правильный прогноз мы получим прибыль = desired_credit_limit * 0.18. 

Так как у нас есть целевая переменная - default_within_1y, то на этом же этапе мы будем определять метрики качества модели. Для этой задачи у нас двоякая ситуация: с одной стороны, если мы допустим ошибку второго рода, то мы получим ощутимый штраф, с другой стороны, если мы допускаем ошибку первого рода, то мы просто не получаем денег, причём заметим, что коэффициент у второй ошибки куда больше, чем у прибыли, поэтому при равенстве кредитного лимита нам куда важнее минимизировать ошибку второго рода. Мы можем пойти по двум сценариям: минимизировать штрафы, использую метрику качества Recall, так как нам важно отловить все дефолты, либо максимизировать выгоду. А можно попробовать совместить в себе два подхода, используя свою метрику:

$$
Q(a, X, y) = \sum_{i=1}^n [\mathbb{I}(a(x^{(i)}) = y_i = 1) * 0 + \mathbb{I}(a(x^{(i)}) =1, y_i = 0) (-0.18 * x^{(i)} _{\text{dcl}}) + \mathbb{I}(a(x^{(i)}) = 0, y_i = 1) (-0.75 * x^{(i)} _{\text{dcl}}) + \mathbb{I}(a(x^{(i)}) = 0, y_i = 0) (0.18 * x^{(i)} _{\text{dcl}})] =
$$

$$
=  \sum_{i=1}^n [\mathbb{I}(a(x^{(i)}) =1, y_i = 0) (-0.18 * x^{(i)} _{\text{dcl}}) + \mathbb{I}(a(x^{(i)}) = 0, y_i = 1) (-0.75 * x^{(i)} _{\text{dcl}}) + \mathbb{I}(a(x^{(i)}) = 0, y_i = 0) (0.18 * x^{(i)} _{\text{dcl}})] \to max_a
$$
Где a - наша модель, X - наши данные, y - истинные метки.

Таким образом, мы будем оценивать нашу модель по абсолютной чистой прибыли.

Минутка занимательной, но не очень применительной математики, которая пришла мне в голову: <details> ecли максимизировать не такой функционал, а поделить его ещё на n, то мы получим эмпирическое среднее. А тогда, если n достаточно большой, то мы можем сказать, что мы максимизируем такое матожидание: 
$\mathbb{E}[\mathbb{I}(a(\xi) = 1, \eta = 0) (-0.18 * \xi _{\text{dcl}}) + \mathbb{I}(a(\xi) = 0, \eta = 1) (-0.75 * \xi _{\text{dcl}}) + \mathbb{I}(a(\xi) = 0, \eta = 0) (0.18 * \xi _{\text{dcl}})]$, а если ещё и если потребовать независимости $\xi _{\text{dcl}}$ и $\eta$, которые, кстати, случайные величины, то вся наша метрика схлопнется в эмпирическую оценку: $Q(a, X, y) = \overline x_{dcl}(0.18\frac{TN}{n} - 0.18 \frac{FP}{n} - 0.75 \frac{FN}{n})$, что круто с точки зрения простоты и наглядности, но для корректности мы сделали слишком сильное допущение о независимости, да и абсолютные значения в нашем случае предпочтительнее относительных, потому что нам интересна абсолютная прибыль. </details>

In [3]:
import numpy as np
def income_metric(y_true: np.ndarray, y_pred: np.ndarray, X):
    x_dcl = np.asarray(X['desired_credit_limit'])
    res = 0 

    res += np.sum(x_dcl[(y_pred == 0) & (y_true == 0) ]) * 0.18
    res -=  np.sum(x_dcl[(y_pred == 1) & (y_true == 0) ]) * 0.18
    res -= np.sum(x_dcl[(y_pred == 0) & (y_true == 1) ]) * 0.75

    return res